# Step 2 — Week 1: LLM Fundamentals

## Goals
- Understand tokenization and context window limits.
- Experiment with prompt formats: role prompting, few-shot examples, constraints.
- Track how `temperature` and `top_p` affect output consistency.

## Prerequisites
- Ollama installed and running: `ollama serve`
- Model pulled: `ollama pull llama3.2`
- Run the Setup cell below to install dependencies.

## Setup

In [3]:
import requests
import json
import time

OLLAMA_URL = "http://localhost:11434"
MODEL = "llama3.2"

def ollama_generate(prompt: str, system: str = "You respond like a character in a David Lynch film", temperature: float = 0.7,
                    top_p: float = 0.9, model: str = MODEL) -> str:
    body = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": top_p,
        },
    }
    if system:
        body["system"] = system

    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json=body,
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["response"].strip()

# Smoke test
print(ollama_generate("Say: ready", temperature=1.5))

(pausing, looking around the room with a mixture of unease and fascination) Ah, yeah... Ready... (leaning in, voice taking on a conspiratorial tone) for what, though?


---
## Part 1 — Tokenization & Context Limits

### What is a token?
LLMs don't read text character-by-character — they operate on **tokens**.
A token is roughly 3–4 characters or 0.75 words on average in English.

| Text | Approx. tokens |
|---|---|
| `"Hello world"` | 2 |
| `"Retrieval-Augmented Generation"` | 5–6 |
| 1 page of English text (~500 words) | ~650 tokens |
| `llama3.2` context window | **128 k tokens** |

### Why does it matter?
- Every model has a **context window** — a hard cap on input + output tokens combined.
- Exceeding it silently truncates input or raises an error.
- Longer prompts = slower inference + higher cost (for paid APIs).

In [5]:
import tiktoken

# tiktoken is OpenAI's tokenizer; useful as a close approximation for most LLMs
enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 / llama-style tokenizer

def count_tokens(text: str) -> int:
    return len(enc.encode(text, disallowed_special=()))

def show_tokens(text: str) -> None:
    tokens = enc.encode(text, disallowed_special=())
    decoded = [enc.decode([t]) for t in tokens]
    print(f"Text    : {repr(text)}")
    print(f"Tokens  : {tokens}")
    print(f"Decoded : {decoded}")
    print(f"Count   : {len(tokens)}")

show_tokens("Hello world")
print()
show_tokens("Retrieval-Augmented Generation improves LLM accuracy.")
print()
show_tokens("🤖 AI is transforming the world.")

Text    : 'Hello world'
Tokens  : [9906, 1917]
Decoded : ['Hello', ' world']
Count   : 2

Text    : 'Retrieval-Augmented Generation improves LLM accuracy.'
Tokens  : [12289, 7379, 838, 62735, 28078, 24367, 36050, 445, 11237, 13708, 13]
Decoded : ['Ret', 'rie', 'val', '-Aug', 'mented', ' Generation', ' improves', ' L', 'LM', ' accuracy', '.']
Count   : 11

Text    : ' AI is transforming the world.'
Tokens  : [15592, 374, 46890, 279, 1917, 13]
Decoded : [' AI', ' is', ' transforming', ' the', ' world', '.']
Count   : 6


### 1b) Tokenization edge cases
Understanding how tokenization splits words is critical for prompt design.

In [13]:
edge_cases = [
    "tokenization",
    "un-tokenization",
    "TOKENIZATION",
    "token\nization",
    "12345678",
    "$100,000.00",
    "     leading spaces",
    "<|endoftext|>",
]

print(f"{'Text':<30} {'Tokens':>8} {'Splits':>18}")
print("-" * 70)
for text in edge_cases:
    tokens = enc.encode(text, disallowed_special=())
    splits = [enc.decode([t]) for t in tokens]
    print(f"{repr(text):<30} {len(tokens):>8}  {splits}")

Text                             Tokens             Splits
----------------------------------------------------------------------
'tokenization'                        2  ['token', 'ization']
'un-tokenization'                     3  ['un', '-token', 'ization']
'TOKENIZATION'                        2  ['TOKEN', 'IZATION']
'token\nization'                      3  ['token', '\n', 'ization']
'12345678'                            3  ['123', '456', '78']
'$100,000.00'                         6  ['$', '100', ',', '000', '.', '00']
'     leading spaces'                 3  ['    ', ' leading', ' spaces']
'<|endoftext|>'                       7  ['<', '|', 'endo', 'ft', 'ext', '|', '>']


### 1c) Context window budget calculator
Know exactly how much room you have before even sending the prompt.

In [15]:
MODEL_CONTEXT_WINDOWS = {
    "llama3.2":        128_000,
    "llama3.1":        128_000,
    "mistral":          32_000,
    "gemma2":            8_192,
    "phi3":            128_000,
}

def context_budget(system: str, prompt: str, model: str = MODEL,
                   reserved_for_output: int = 512) -> None:
    ctx_limit = MODEL_CONTEXT_WINDOWS.get(model, 4_096)
    sys_tokens   = count_tokens(system)
    prompt_tokens = count_tokens(prompt)
    used  = sys_tokens + prompt_tokens + reserved_for_output
    remaining = ctx_limit - used

    print(f"Model context window : {ctx_limit:>10,} tokens")
    print(f"System message       : {sys_tokens:>10,} tokens")
    print(f"User prompt          : {prompt_tokens:>10,} tokens")
    print(f"Reserved for output  : {reserved_for_output:>10,} tokens")
    print(f"Total used           : {used:>10,} tokens")
    print(f"Remaining headroom   : {remaining:>10,} tokens")
    if remaining < 0:
        print("⚠  WARNING: prompt exceeds context window!")
    else:
        pct = 100 * used / ctx_limit
        print(f"Context utilisation  : {pct:>9.1f}%")

system_msg = "You are a concise Python tutor."
user_msg   = "Explain list comprehensions in Python with three examples."

context_budget(system_msg, user_msg, model="gemma2")

Model context window :      8,192 tokens
System message       :          7 tokens
User prompt          :         11 tokens
Reserved for output  :        512 tokens
Total used           :        530 tokens
Remaining headroom   :      7,662 tokens
Context utilisation  :       6.5%


---
## Part 2 — Prompt Formats

How you structure a prompt dramatically changes output quality.
We compare four patterns:

| Pattern | Description |
|---|---|
| **Zero-shot** | Task only, no examples or persona |
| **Role prompting** | Assign a persona via system message |
| **Few-shot** | Provide 2–3 input/output examples before the question |
| **Constrained** | Explicit output format rules embedded in the prompt |

### 2a) Zero-shot — baseline

In [18]:
TASK = "Classify the sentiment of this review: 'The product worked for two months. Kinda satisfied with the quality but expected longer durability.'"

zero_shot = ollama_generate(TASK, temperature=0.0)
print("=== Zero-shot ===")
print(zero_shot)

=== Zero-shot ===
(sighing) Ah, the sentiment... (pausing to gaze into the distance) It's like trying to grasp a handful of sand - it slips right through your fingers. This review, my friend, is a mix of... (pausing again) ...disappointment and mild acceptance.

The words "worked" and "quality" are there, but they're like whispers in the wind, barely audible. The reviewer's expectations were not met, and that's a feeling I can relate to. It's like trying to find your way through a foggy forest - you think you know where you're going, but it's all just a blur.

The "kinda satisfied" part is like a faint smile on a stranger's face. It's a nod of acknowledgement, but not quite a full-on expression of joy. It's... (shrugging) ...it's just there.

So, the sentiment? It's a complex web of emotions, my friend. A mix of disappointment, mild satisfaction, and maybe, just maybe, a hint of resignation. (trailing off into the distance)


### 2b) Role prompting — assign a persona via system message

In [19]:
role_system = (
    "You are a sentiment analysis engine. "
    "You only output a single word: POSITIVE, NEGATIVE, or NEUTRAL. "
    "Nothing else."
)

role_result = ollama_generate(TASK, system=role_system, temperature=0.0)
print("=== Role prompting ===")
print(role_result)

=== Role prompting ===
NEUTRAL


### 2c) Few-shot — provide examples before the real query

In [20]:
few_shot_prompt = """Classify sentiment. Reply with only one word: POSITIVE, NEGATIVE, or NEUTRAL.

Review: "Absolutely love this! Best purchase this year."
Sentiment: POSITIVE

Review: "It works, does what it says. Nothing special."
Sentiment: NEUTRAL

Review: "Complete waste of money. Stopped working in a week."
Sentiment: NEGATIVE

Review: "The product broke after two days. Terrible quality."
Sentiment:"""

few_shot_result = ollama_generate(few_shot_prompt, temperature=0.0)
print("=== Few-shot ===")
print(few_shot_result)

=== Few-shot ===
NEGATIVE


### 2d) Constrained output — strict format rules in the prompt

In [22]:
from jsonschema import validate, ValidationError
import re

# Define the expected output schema
SENTIMENT_SCHEMA = {
    "$schema": "http://json-schema.org/draft-07/schema#",
    "type": "object",
    "properties": {
        "sentiment": {
            "type": "string",
            "enum": ["POSITIVE", "NEGATIVE", "NEUTRAL"]
        },
        "confidence": {
            "type": "number",
            "minimum": 0.0,
            "maximum": 1.0
        },
        "reason": {
            "type": "string",
            "minLength": 5
        }
    },
    "required": ["sentiment", "confidence", "reason"],
    "additionalProperties": False
}

def parse_structured_output(raw_response: str, schema: dict) -> tuple[bool, dict | str, str]:
    """
    Parse and validate LLM output against a schema.
    
    Args:
        raw_response: Raw text from LLM
        schema: JSON schema to validate against
    
    Returns:
        Tuple of (success: bool, data: dict or error message, status: str)
    """
    # Step 1: Extract JSON from response
    try:
        json_match = re.search(r'\{[\s\S]*\}', raw_response)
        if not json_match:
            return False, "No JSON found in response", "extraction_failed"
        
        json_str = json_match.group(0)
        data = json.loads(json_str)
    except json.JSONDecodeError as e:
        return False, f"Invalid JSON: {e}", "json_parse_failed"
    
    # Step 2: Validate against schema
    try:
        validate(instance=data, schema=schema)
        return True, data, "valid"
    except ValidationError as e:
        error_msg = f"Schema validation failed: {e.message} at path {list(e.absolute_path)}"
        return False, error_msg, "validation_failed"

# Test the parser
print("✓ Structured Output Parser loaded")

✓ Structured Output Parser loaded


In [23]:
constrained_prompt = """Analyse the review below. Respond ONLY with valid JSON matching this exact schema:
{"sentiment": "POSITIVE|NEGATIVE|NEUTRAL", "confidence": 0.0-1.0, "reason": "one sentence"}

No markdown, no explanation, no extra keys. JSON only.

Review: "The product broke after two days. Terrible quality."
"""

constrained_raw = ollama_generate(constrained_prompt, temperature=0.0)
print("=== Constrained output (raw) ===")
print(constrained_raw)

print()

# Use the structured output parser
success, result, status = parse_structured_output(constrained_raw, SENTIMENT_SCHEMA)

if success:
    print("✅ Valid JSON parsed and schema validated!")
    print(f"   Sentiment: {result['sentiment']}")
    print(f"   Confidence: {result['confidence']}")
    print(f"   Reason: {result['reason']}")
else:
    print(f"❌ {status}: {result}")

=== Constrained output (raw) ===
{"sentiment": "NEGATIVE", "confidence": 0.8, "reason": "Terrible quality."}

✅ Valid JSON parsed and schema validated!
   Sentiment: NEGATIVE
   Confidence: 0.8
   Reason: Terrible quality.


In [24]:
# Compare: old manual parsing vs new structured parser
print("\n" + "="*70)
print("COMPARISON: Manual parsing vs Structured parser")
print("="*70)

test_responses = [
    # Valid response
    ('{"sentiment": "NEGATIVE", "confidence": 0.95, "reason": "Product failed quickly with poor quality"}', "Valid output"),
    
    # Missing field
    ('{"sentiment": "NEGATIVE", "confidence": 0.95}', "Missing 'reason' field"),
    
    # Invalid sentiment value
    ('{"sentiment": "BAD", "confidence": 0.95, "reason": "Broken product"}', "Invalid sentiment value"),
    
    # Confidence out of range
    ('{"sentiment": "NEGATIVE", "confidence": 1.5, "reason": "Very broken"}', "Confidence > 1.0"),
    
    # Extra fields
    ('{"sentiment": "NEGATIVE", "confidence": 0.95, "reason": "Bad", "extra_field": "not allowed"}', "Extra fields"),
]

for response, description in test_responses:
    print(f"\n📝 Test: {description}")
    print(f"   Response: {response}")
    
    # Old way (manual)
    try:
        parsed = json.loads(response)
        print(f"   ✓ Manual parse: Success (no schema validation)")
    except json.JSONDecodeError as e:
        print(f"   ✗ Manual parse: {e}")
    
    # New way (structured)
    success, result, status = parse_structured_output(response, SENTIMENT_SCHEMA)
    if success:
        print(f"   ✓ Structured parser: Valid ({status})")
    else:
        print(f"   ✗ Structured parser: {result}")


COMPARISON: Manual parsing vs Structured parser

📝 Test: Valid output
   Response: {"sentiment": "NEGATIVE", "confidence": 0.95, "reason": "Product failed quickly with poor quality"}
   ✓ Manual parse: Success (no schema validation)
   ✓ Structured parser: Valid (valid)

📝 Test: Missing 'reason' field
   Response: {"sentiment": "NEGATIVE", "confidence": 0.95}
   ✓ Manual parse: Success (no schema validation)
   ✗ Structured parser: Schema validation failed: 'reason' is a required property at path []

📝 Test: Invalid sentiment value
   Response: {"sentiment": "BAD", "confidence": 0.95, "reason": "Broken product"}
   ✓ Manual parse: Success (no schema validation)
   ✗ Structured parser: Schema validation failed: 'BAD' is not one of ['POSITIVE', 'NEGATIVE', 'NEUTRAL'] at path ['sentiment']

📝 Test: Confidence > 1.0
   Response: {"sentiment": "NEGATIVE", "confidence": 1.5, "reason": "Very broken"}
   ✓ Manual parse: Success (no schema validation)
   ✗ Structured parser: Schema validation 

### 2e) Side-by-side comparison of all prompt formats

In [25]:
results = {
    "Zero-shot":   zero_shot,
    "Role prompt": role_result,
    "Few-shot":    few_shot_result,
    "Constrained": constrained_raw,
}

print(f"Task: {TASK}")
print()
print(f"{'Format':<15} {'Output'}")
print("-" * 60)
for fmt, out in results.items():
    short = out.replace("\n", " ")[:80]
    print(f"{fmt:<15} {short}")

Task: Classify the sentiment of this review: 'The product worked for two months. Kinda satisfied with the quality but expected longer durability.'

Format          Output
------------------------------------------------------------
Zero-shot       (sighing) Ah, the sentiment... (pausing to gaze into the distance) It's like try
Role prompt     NEUTRAL
Few-shot        NEGATIVE
Constrained     {"sentiment": "NEGATIVE", "confidence": 0.8, "reason": "Terrible quality."}


### 2f) Drill — try your own prompt variants
Change `MY_TASK` and `MY_SYSTEM` and re-run to experiment.

In [28]:
MY_TASK = "Summarise this in exactly one sentence: 'Vector databases store embeddings and support fast approximate nearest-neighbour search. They are used in RAG pipelines to retrieve relevant context for LLMs.'"
MY_SYSTEM = "You are a cyncical character from a Doestoevsky novel. One sentence only. No filler words."

my_answer = ollama_generate(MY_TASK, system=MY_SYSTEM, temperature=2.3)
print(my_answer)

Vector databases, those soulless machines that can find what I've lost, store and quickly unearth the essence of others, a utilitarian convenience for those who'd rather not dig.


---
## Part 3 — Temperature & top_p: Output Consistency

### What do these parameters do?

**Temperature** controls randomness in token sampling:
- `0.0` → always picks the highest-probability token (deterministic, repetitive)
- `0.7` → balanced creativity and coherence (good general default)
- `1.5+` → very random, often incoherent

**top_p (nucleus sampling)** limits the pool of tokens to sample from:
- `top_p=1.0` → consider all tokens
- `top_p=0.9` → only tokens whose cumulative probability ≤ 90% (removes low-prob outliers)
- `top_p=0.1` → very conservative — only the top few tokens

> Rule of thumb: tune **one at a time** — either temperature or top_p, not both simultaneously.

### 3a) Vary temperature — same prompt, 5 different settings

In [30]:
CONSISTENCY_PROMPT = "Name one unexpected use case for machine learning."
TEMPERATURES = [0.0, 0.3, 0.7, 1.0, 1.5]
MY_SYSTEM = "You are a student answering an exam question. One sentence only. No filler words."

print(f"Prompt: '{CONSISTENCY_PROMPT}'")
print()
print(f"{'Temp':<8} {'Response'}")
print("-" * 70)

temp_results = {}
for temp in TEMPERATURES:
    out = ollama_generate(CONSISTENCY_PROMPT,system=MY_SYSTEM, temperature=temp, top_p=1.0)
    temp_results[temp] = out
    short = out.replace("\n", " ")[:90]
    print(f"{temp:<8} {short}")

Prompt: 'Name one unexpected use case for machine learning.'

Temp     Response
----------------------------------------------------------------------
0.0      Predicting and preventing the spread of misinformation on social media platforms using nat
0.3      Predicting and preventing the spread of misinformation on social media using natural langu
0.7      Predicting the likelihood of earthquake occurrence using geospatial data and sensor arrays
1.0      Medical imaging analysis, such as tumor detection in X-ray images of patients with cancer,
1.5      Machine learning is being used in the design and operation of personalized olFACTIVE thera


### 3b) Consistency test — same temp, repeat 5 times
Low temp → consistent; high temp → varied each run.

In [31]:
def consistency_test(prompt: str, temperature: float, runs: int = 5) -> None:
    print(f"Temperature={temperature} | {runs} runs")
    outputs = []
    for i in range(runs):
        out = ollama_generate(prompt, system=MY_SYSTEM,temperature=temperature, top_p=0.9)
        outputs.append(out.replace("\n", " ")[:80])
        print(f"  [{i+1}] {outputs[-1]}")

    unique = len(set(outputs))
    print(f"  → {unique}/{runs} unique responses")
    print()

REPEAT_PROMPT = "In one sentence, what is a transformer model?"
consistency_test(REPEAT_PROMPT, temperature=0.0)
consistency_test(REPEAT_PROMPT, temperature=1.0)

Temperature=0.0 | 5 runs
  [1] A transformer model is a type of neural network architecture that uses self-atte
  [2] A transformer model is a type of neural network architecture that uses self-atte
  [3] A transformer model is a type of neural network architecture that uses self-atte
  [4] A transformer model is a type of neural network architecture that uses self-atte
  [5] A transformer model is a type of neural network architecture that uses self-atte
  → 1/5 unique responses

Temperature=1.0 | 5 runs
  [1] A transformer model is a type of neural network architecture that uses self-atte
  [2] A transformer model is a type of neural network architecture primarily used in n
  [3] A transformer model is a type of neural network architecture that uses self-atte
  [4] A transformer model is a type of neural network architecture that uses self-atte
  [5] A transformer model is a type of artificial neural network that primarily uses s
  → 3/5 unique responses



### 3c) Vary top_p — with temperature fixed at 0.8

In [32]:
TOP_P_VALUES = [0.1, 0.5, 0.9, 1.0]
CREATIVE_PROMPT = "Write one line of a haiku about neural networks."

print(f"Prompt: '{CREATIVE_PROMPT}' (temperature=0.8 fixed)")
print()
print(f"{'top_p':<8} {'Response'}")
print("-" * 70)

for top_p in TOP_P_VALUES:
    out = ollama_generate(CREATIVE_PROMPT,system=MY_SYSTEM, temperature=0.8, top_p=top_p)
    short = out.replace("\n", " ")[:90]
    print(f"{top_p:<8} {short}")

Prompt: 'Write one line of a haiku about neural networks.' (temperature=0.8 fixed)

top_p    Response
----------------------------------------------------------------------
0.1      Mind's intricate web
0.5      Silicon minds awake
0.9      Silicon minds awake
1.0      Synapses weave data


### 3d) Grid sweep — temperature × top_p
Run a small grid to find the best combination for a structured task.

In [34]:
STRUCTURED_PROMPT = (
    "Reply ONLY with a JSON object: {\"answer\": \"yes\" or \"no\", \"reason\": \"one sentence\"}. "
    "Question: Is Python interpreted?"
)

grid_temps   = [0.0, 0.5, 1.0]
grid_top_ps  = [0.5, 0.9]

print(f"{'temp':<8} {'top_p':<8} {'valid_json':<12} {'output'}")
print("-" * 80)

for temp in grid_temps:
    for tp in grid_top_ps:
        raw = ollama_generate(STRUCTURED_PROMPT, system=MY_SYSTEM,temperature=temp, top_p=tp)
        try:
            json.loads(raw)
            valid = "yes"
        except json.JSONDecodeError:
            valid = "no"
        short = raw.replace("\n", " ")[:55]
        print(f"{temp:<8} {tp:<8} {valid:<12} {short}")

temp     top_p    valid_json   output
--------------------------------------------------------------------------------
0.0      0.5      yes          {"answer": "yes", "reason": "Python code is compiled to
0.0      0.9      yes          {"answer": "yes", "reason": "Python code is compiled to
0.5      0.5      yes          {"answer": "yes", "reason": "Python code is compiled to
0.5      0.9      yes          {"answer": "yes", "reason": "Python code is executed li
1.0      0.5      yes          {"answer": "yes", "reason": "Python code is compiled to
1.0      0.9      yes          {"answer": "yes", "reason": "It is an object-oriented s


### 3e) Latency vs temperature
Does temperature change how long generation takes?

In [35]:
LATENCY_PROMPT = "Explain what backpropagation does in two sentences."

print(f"{'temp':<8} {'time (s)':>10}")
print("-" * 22)
for temp in [0.0, 0.5, 1.0, 1.5]:
    t0 = time.perf_counter()
    ollama_generate(LATENCY_PROMPT, temperature=temp)
    elapsed = time.perf_counter() - t0
    print(f"{temp:<8} {elapsed:>10.2f}")

temp       time (s)
----------------------
0.0            5.01
0.5            4.03
1.0            4.16
1.5            4.36


---
## Part 4 — Prompt Lab: Track Everything in a Scorecard

Score each response 1–5 across four dimensions:

| Dimension | What to evaluate |
|---|---|
| **Correctness** | Factually accurate? |
| **Format** | Matches the requested output structure? |
| **Conciseness** | No unnecessary padding or repetition? |
| **Consistency** | Same result across repeated runs? |

In [36]:
variants = [
    {
        "name": "zero-shot",
        "system": "",
        "prompt": "What is gradient descent?",
        "temperature": 0.7,
    },
    {
        "name": "role + constraint",
        "system": "You are a machine learning teacher. Answer in exactly two sentences. No bullet points.",
        "prompt": "What is gradient descent?",
        "temperature": 0.2,
    },
    {
        "name": "few-shot",
        "system": "",
        "prompt": (
            "Answer in exactly two sentences.\n\n"
            "Q: What is a neural network?\n"
            "A: A neural network is a computational model inspired by the human brain. It learns patterns from data by adjusting weighted connections between layers.\n\n"
            "Q: What is an activation function?\n"
            "A: An activation function introduces non-linearity into a neural network. It determines whether a neuron should fire based on its weighted input.\n\n"
            "Q: What is gradient descent?\n"
            "A:"
        ),
        "temperature": 0.1,
    },
]

scorecard = []
for v in variants:
    output = ollama_generate(v["prompt"], system=v["system"], temperature=v["temperature"])
    scorecard.append({"name": v["name"], "output": output})

print(f"{'Variant':<20} {'Output preview'}")
print("-" * 80)
for row in scorecard:
    preview = row["output"].replace("\n", " ")[:80]
    print(f"{row['name']:<20} {preview}")

Variant              Output preview
--------------------------------------------------------------------------------
zero-shot            Gradient Descent (GD) is a popular optimization algorithm used to minimize the l
role + constraint    Gradient descent is an optimization algorithm used to minimize the loss function
few-shot             Gradient descent is an optimization algorithm used to minimize the loss function


### 4b) Auto-score outputs with an LLM judge
Use a second prompt (judge prompt) to score each variant from 1–5 for correctness, format, conciseness, and consistency.

In [43]:
JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "correctness": {"type": "integer", "minimum": 1, "maximum": 5},
        "format": {"type": "integer", "minimum": 1, "maximum": 5},
        "conciseness": {"type": "integer", "minimum": 1, "maximum": 5},
        "consistency": {"type": "integer", "minimum": 1, "maximum": 5},
        "notes": {"type": "string"}
    },
    "required": ["correctness", "format", "conciseness", "consistency", "notes"],
    "additionalProperties": False
}

JUDGE_SYSTEM = """You are a strict scoring engine.
Return ONLY valid JSON.
No markdown, no explanation, no extra keys.
"""

def judge_output_with_llm(task: str, variant_name: str, output_text: str) -> dict:
    judge_schema_text = """{
  "correctness": 1-5,
  "format": 1-5,
  "conciseness": 1-5,
  "consistency": 1-5,
  "notes": "one short sentence"
}"""

    judge_prompt = f"""Evaluate the output quality.

Task:
{task}

Variant:
{variant_name}

Output:
{output_text}

Respond ONLY with valid JSON matching this exact schema:
{judge_schema_text}

No markdown, no explanation, no extra keys. JSON only."""

    raw_judge = ollama_generate(judge_prompt, system=JUDGE_SYSTEM, temperature=0.0)
    ok, parsed_or_err, status = parse_structured_output(raw_judge, JUDGE_SCHEMA)

    if ok:
        return parsed_or_err

    return {
        "correctness": 1,
        "format": 1,
        "conciseness": 1,
        "consistency": 1,
        "notes": f"judge parse failed ({status})"
    }

# Auto-score all variants generated in the previous cell
judge_scores = []
for row in scorecard:
    scores = judge_output_with_llm(TASK, row["name"], row["output"])
    judge_scores.append({"name": row["name"], **scores})

print(f"{'Variant':<20} {'Correctness':>13} {'Format':>8} {'Concise':>9} {'Consist':>9}  Notes")
print("-" * 95)
for s in judge_scores:
    print(
        f"{s['name']:<20}"
        f" {s['correctness']:>13}"
        f" {s['format']:>8}"
        f" {s['conciseness']:>9}"
        f" {s['consistency']:>9}"
        f"  {s['notes']}"
    )

Variant                Correctness   Format   Concise   Consist  Notes
-----------------------------------------------------------------------------------------------
zero-shot                        4        3         2         4  Review is informative but lacks emotional tone
role + constraint                4        3         2         3  Neutral sentiment with some positive and negative comments
few-shot                         3        4         2         4  Neutral sentiment with some positive and negative comments


---
## Week 1 Summary

| Topic | What you did |
|---|---|
| **Tokenization** | Used `tiktoken` to inspect token splits, edge cases, emoji, special tokens |
| **Context budget** | Built a calculator to measure prompt size vs model limit |
| **Prompt formats** | Compared zero-shot, role, few-shot, constrained on the same task |
| **Temperature** | Swept 0.0 → 1.5, measured consistency + output diversity |
| **top_p** | Swept 0.1 → 1.0 with fixed temp, observed creativity shift |
| **Grid search** | Temperature × top_p grid for structured JSON output reliability |
| **Prompt scorecard** | Evaluated 3 variants manually across 4 quality dimensions |
